In [ ]:
%pip install -q diffusers accelerate transformers kagglehub bitsandbytes sentencepiece protobuf

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
import os, gc, math, random, logging
from pathlib import Path

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from tqdm.auto import tqdm

from accelerate import Accelerator
from accelerate.logging import get_logger
from accelerate.utils import ProjectConfiguration

from transformers import (
    CLIPTokenizer, CLIPTextModelWithProjection,
    T5TokenizerFast, T5EncoderModel,
    get_cosine_schedule_with_warmup,
)
from diffusers import (
    SD3Transformer2DModel,
    FlowMatchEulerDiscreteScheduler,
    AutoencoderKL,
    StableDiffusion3Pipeline,
)


In [ ]:
def add_flow_noise(latents, noise, timesteps, num_train_timesteps=1000):
    t = (timesteps.float() / num_train_timesteps).view(-1,1,1,1).to(latents.device)
    t = t.to(latents.dtype)
    return ((1 - t) * latents + t * noise).to(latents.dtype)
import kagglehub

In [ ]:
MODEL_ID            = "stabilityai/stable-diffusion-3-medium-diffusers"
OUTPUT_DIR          = "/content/sd3-finetuned"
KAGGLE_DATASET      = "yashraj2311/chimera-dataset"
NUM_IMAGES          = 1200
RESOLUTION          = 512
TRAIN_BATCH_SIZE    = 1
GRADIENT_ACCUM      = 1
NUM_EPOCHS          = 10
LEARNING_RATE       = 5e-6
LR_WARMUP_STEPS     = 40
MIXED_PRECISION     = "bf16"
GRADIENT_CKPT       = True
SAVE_EVERY_N_EPOCHS = 2
SEED                = 42
MAX_GRAD_NORM       = 1.0
USE_8BIT_ADAM       = True

logging.basicConfig(level=logging.INFO)
logger = get_logger(__name__, log_level="INFO")

In [ ]:
def free_memory(*objects):
    """Delete objects and flush GPU cache."""
    for obj in objects:
        del obj
    gc.collect()
    torch.cuda.empty_cache()


def vram_usage(label=""):
    used = torch.cuda.memory_allocated() / 1e9
    reserved = torch.cuda.memory_reserved() / 1e9
    print(f"[VRAM {label}] allocated {used:.2f} GB | reserved {reserved:.2f} GB")

In [ ]:
class EmbeddingDataset(Dataset):
    def __init__(self, latents, prompt_embeds, pooled_embeds):
        self.latents = latents
        self.prompt_embeds = prompt_embeds
        self.pooled_embeds = pooled_embeds

    def __len__(self):
        return len(self.latents)

    def __getitem__(self, idx):
        return {
            "latents": self.latents[idx],
            "prompt_embeds": self.prompt_embeds[idx],
            "pooled_embeds": self.pooled_embeds[idx],
        }

def load_kaggle_dataset(dataset_name, num_images, seed):
    logger.info(f"Downloading Kaggle dataset: {dataset_name}")
    dataset_path = kagglehub.dataset_download(dataset_name)
    base = Path(dataset_path)

    image_exts = {".jpg", ".jpeg", ".png", ".webp"}
    all_images = sorted(p for p in base.rglob("*") if p.suffix.lower() in image_exts)
    logger.info(f"Found {len(all_images)} images.")

    random.seed(seed)
    selected = random.sample(all_images, min(num_images, len(all_images)))

    # captions
    caption_map = {}

    import csv

    for cap_file in list(base.rglob("captions.txt")) + list(base.rglob("*.csv")):
        logger.info(f"Trying caption file: {cap_file}")
        try:
            with open(cap_file, newline="", encoding="utf-8") as f:
                # Sniff the delimiter
                sample = f.read(2048); f.seek(0)
                try:
                    dialect = csv.Sniffer().sniff(sample, delimiters=",\t")
                except csv.Error:
                    dialect = csv.excel   # fallback: comma

                reader = csv.reader(f, dialect)
                header = next(reader, None)

                # Detect column indices
                if header:
                    header_lower = [h.lower().strip() for h in header]
                    img_idx = next((i for i, h in enumerate(header_lower)
                                    if "image" in h or "file" in h), 0)
                    cap_idx = next((i for i, h in enumerate(header_lower)
                                    if "caption" in h or "comment" in h
                                    or "description" in h), 1)
                else:
                    img_idx, cap_idx = 0, 1   # no header: image,caption

                for row in reader:
                    if len(row) <= max(img_idx, cap_idx):
                        continue
                    key = Path(row[img_idx].strip()).name
                    cap = row[cap_idx].strip()
                    if key and cap and key not in caption_map:
                        caption_map[key] = cap

        except Exception as e:
            logger.warning(f"  Could not parse {cap_file}: {e}")
            continue

        if caption_map:
            logger.info(f"  Loaded {len(caption_map)} captions from {cap_file.name}")
            break

    image_paths, captions = [], []
    for p in selected:
        cap = caption_map.get(p.name) or caption_map.get(p.stem) or f"a photo of {p.stem.replace('_',' ')}"
        image_paths.append(str(p))
        captions.append(cap)

    logger.info(f"Using {len(image_paths)} pairs.")
    for i in range(min(3, len(captions))):
        logger.info(f"  [{i}] {Path(image_paths[i]).name} → \"{captions[i][:80]}\"")
    return image_paths, captions



In [ ]:
%pip install matplotlib

import matplotlib.pyplot as plt
from PIL import Image
from accelerate import Accelerator

accelerator = Accelerator(mixed_precision="bf16")

image_paths, captions = load_kaggle_dataset(KAGGLE_DATASET, num_images=5, seed=SEED)

# Display the first 5 images
for i in range(min(5, len(image_paths))):
    img = Image.open(image_paths[i])
    plt.figure(figsize=(8, 5))
    plt.imshow(img)
    plt.title(f"Caption: {captions[i]}", fontsize=10, wrap=True)
    plt.axis('off')
    plt.show()

In [ ]:
IMG_TRANSFORM = transforms.Compose([
    transforms.Resize(RESOLUTION, interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.CenterCrop(RESOLUTION),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5]),
])


@torch.no_grad()
def precompute_embeddings(image_paths, captions, device):
    """
    Encode all images (VAE) and captions (CLIP×2 + T5) to tensors,
    store on CPU. Returns lists ready to build EmbeddingDataset.
    """
    weight_dtype = torch.float16   # T4 is fp16 only

    # load frozen encoders
    logger.info("Loading VAE …")
    vae = AutoencoderKL.from_pretrained(MODEL_ID, subfolder="vae",
                                        torch_dtype=weight_dtype).to(device)
    vae.eval()

    logger.info("Loading CLIP encoders …")
    clip_tok_l  = CLIPTokenizer.from_pretrained(MODEL_ID, subfolder="tokenizer")
    clip_tok_g  = CLIPTokenizer.from_pretrained(MODEL_ID, subfolder="tokenizer_2")
    clip_enc_l  = CLIPTextModelWithProjection.from_pretrained(
                        MODEL_ID, subfolder="text_encoder",
                        torch_dtype=weight_dtype).to(device)
    clip_enc_g  = CLIPTextModelWithProjection.from_pretrained(
                        MODEL_ID, subfolder="text_encoder_2",
                        torch_dtype=weight_dtype).to(device)
    clip_enc_l.eval(); clip_enc_g.eval()

    # T5 on GPU (we now have enough VRAM)
    logger.info("Loading T5 encoder on GPU …")

    t5_tok = T5TokenizerFast.from_pretrained(MODEL_ID, subfolder="tokenizer_3")

    t5_enc = T5EncoderModel.from_pretrained(
        MODEL_ID,
        subfolder="text_encoder_3",
        torch_dtype=weight_dtype
    ).to(device)

    t5_enc.eval()

    vram_usage("after T5 load")

    all_latents, all_prompt_embeds, all_pooled = [], [], []

    for img_path, caption in tqdm(zip(image_paths, captions),
                                   total=len(image_paths), desc="Pre-computing"):
        #  VAE
        img = Image.open(img_path).convert("RGB")
        pixel = IMG_TRANSFORM(img).unsqueeze(0).to(device, dtype=weight_dtype)
        latent = vae.encode(pixel).latent_dist.sample() * vae.config.scaling_factor
        all_latents.append(latent.squeeze(0).cpu())

        #  CLIP-L
        def clip_encode(tok, enc, text):
            toks = tok(text, padding="max_length", max_length=77,
                        truncation=True, return_tensors="pt").to(device)
            out  = enc(**toks, output_hidden_states=True)
            return out.hidden_states[-2], out.text_embeds   # [1,77,D], [1,D]

        h_l, p_l = clip_encode(clip_tok_l, clip_enc_l, caption)
        h_g, p_g = clip_encode(clip_tok_g, clip_enc_g, caption)
        clip_hidden = torch.cat([h_l, h_g], dim=-1)   # [1,77,2048]

        # T5 (CPU inference, result moved to CPU store)
        t5_toks = t5_tok(caption, padding="max_length", max_length=256,
                          truncation=True, return_tensors="pt").to(device)  # move to GPU
        t5_hidden = t5_enc(**t5_toks).last_hidden_state.cpu()  # result back to CPU          # [1,256,4096] CPU

        # pad CLIP to 4096 and concat along sequence dim
        clip_pad = F.pad(clip_hidden.cpu(), (0, 4096 - clip_hidden.shape[-1]))  # [1,77,4096]
        prompt_embeds = torch.cat([clip_pad, t5_hidden], dim=1)    # [1,333,4096]
        pooled_embeds = torch.cat([p_l, p_g], dim=-1)              # [1,2048]

        all_prompt_embeds.append(prompt_embeds.squeeze(0).cpu())
        all_pooled.append(pooled_embeds.squeeze(0).cpu())

    # free all frozen encoders from GPU
    logger.info("Freeing encoders from GPU …")
    free_memory(vae, clip_enc_l, clip_enc_g, t5_enc,
                clip_tok_l, clip_tok_g, t5_tok)
    vram_usage("after encoder cleanup")

    return all_latents, all_prompt_embeds, all_pooled



In [ ]:
def train():
    torch.manual_seed(SEED)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    free_memory()

    
    image_paths, captions = load_kaggle_dataset(KAGGLE_DATASET, NUM_IMAGES, SEED)
    all_latents, all_prompt_embeds, all_pooled = precompute_embeddings(
        image_paths, captions, device
    )
    
    vram_usage("after precompute — encoders freed")

    
    dataset = EmbeddingDataset(all_latents, all_prompt_embeds, all_pooled)
    dataloader = DataLoader(
        dataset,
        batch_size=TRAIN_BATCH_SIZE,
        shuffle=True,
        num_workers=4,
        pin_memory=True,
    )

    
    project_cfg = ProjectConfiguration(project_dir=OUTPUT_DIR,
                                       logging_dir=os.path.join(OUTPUT_DIR, "logs"))
    accelerator = Accelerator(
        gradient_accumulation_steps=GRADIENT_ACCUM,
        mixed_precision="bf16",
        project_config=project_cfg,
    )

    os.makedirs(OUTPUT_DIR, exist_ok=True)
    logger.info("Loading SD3 transformer …")
    transformer = SD3Transformer2DModel.from_pretrained(
        MODEL_ID, subfolder="transformer", torch_dtype=torch.bfloat16,
    ).to(device)
    transformer.enable_gradient_checkpointing()
    vram_usage("after transformer load")

    noise_scheduler = FlowMatchEulerDiscreteScheduler.from_pretrained(
        MODEL_ID, subfolder="scheduler"
    )

    # optimizer
    if USE_8BIT_ADAM:
        import bitsandbytes as bnb
        optimizer = bnb.optim.AdamW8bit(
            transformer.parameters(), lr=LEARNING_RATE,
            betas=(0.9, 0.999), weight_decay=1e-2, eps=1e-8,
        )
    else:
        optimizer = torch.optim.AdamW(
            transformer.parameters(), lr=LEARNING_RATE,
            betas=(0.9, 0.999), weight_decay=1e-2, eps=1e-8,
        )

    num_update_steps = math.ceil(len(dataloader) / GRADIENT_ACCUM) * NUM_EPOCHS
    lr_scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=LR_WARMUP_STEPS,
        num_training_steps=num_update_steps,
    )

    transformer, optimizer, dataloader, lr_scheduler = accelerator.prepare(
        transformer, optimizer, dataloader, lr_scheduler
    )
    vram_usage("after accelerate prepare")

    # ── Training loop 
    global_step = 0
    for epoch in range(NUM_EPOCHS):
        transformer.train()
        epoch_loss = 0.0
        pbar = tqdm(total=len(dataloader), desc=f"Epoch {epoch+1}/{NUM_EPOCHS}")

        for step, batch in enumerate(dataloader):
            with accelerator.accumulate(transformer):
                # tensors come pre-computed — just move to GPU
                latents_b       = batch["latents"].to(device, dtype=torch.bfloat16)
                prompt_embeds_b = batch["prompt_embeds"].to(device, dtype=torch.bfloat16)
                pooled_embeds_b = batch["pooled_embeds"].to(device, dtype=torch.bfloat16)

                bsz = latents_b.shape[0]
                timesteps = torch.randint(
                    0, noise_scheduler.config.num_train_timesteps,
                    (bsz,), device=device,
                ).long()

                noise         = torch.randn_like(latents_b)
                noisy_latents = add_flow_noise(
                    latents_b, noise, timesteps,
                    num_train_timesteps=noise_scheduler.config.num_train_timesteps,
                )

                model_pred = transformer(
                    hidden_states         = noisy_latents,
                    timestep              = timesteps,
                    encoder_hidden_states = prompt_embeds_b,
                    pooled_projections    = pooled_embeds_b,
                    return_dict           = False,
                )[0]

                target = noise - latents_b
                loss   = F.mse_loss(model_pred.float(), target.float(), reduction="mean")

                epoch_loss += loss.detach().item()
                accelerator.backward(loss)

                if accelerator.sync_gradients:
                    accelerator.clip_grad_norm_(transformer.parameters(), MAX_GRAD_NORM)

                optimizer.step()
                lr_scheduler.step()
                optimizer.zero_grad(set_to_none=True)

            if accelerator.sync_gradients:
                global_step += 1

            pbar.update(1)
            pbar.set_postfix(loss=f"{loss.detach().item():.4f}")

        pbar.close()
        avg = epoch_loss / len(dataloader)
        logger.info(f"Epoch {epoch+1}/{NUM_EPOCHS} — avg loss: {avg:.4f}")

        if (epoch + 1) % SAVE_EVERY_N_EPOCHS == 0:
            ckpt = os.path.join(OUTPUT_DIR, f"checkpoint-epoch-{epoch+1}")
            accelerator.unwrap_model(transformer).save_pretrained(ckpt)
            logger.info(f"Saved checkpoint → {ckpt}")

    accelerator.wait_for_everyone()
    pipeline = StableDiffusion3Pipeline.from_pretrained(
        MODEL_ID,
        transformer=accelerator.unwrap_model(transformer),
        torch_dtype=torch.float16,
    )
    pipeline.save_pretrained(OUTPUT_DIR)
    logger.info(f"Done! Pipeline saved to {OUTPUT_DIR}")
    accelerator.end_training()

In [ ]:
if __name__ == "__main__":
    train()

In [ ]:
import torch
from diffusers import StableDiffusion3Pipeline, SD3Transformer2DModel

MODEL_ID = "stabilityai/stable-diffusion-3-medium-diffusers"
CHECKPOINT = "/content/sd3-finetuned/checkpoint-epoch-10"

# load finetuned transformer
transformer = SD3Transformer2DModel.from_pretrained(
    CHECKPOINT,
    torch_dtype=torch.float16
)

pipe = StableDiffusion3Pipeline.from_pretrained(
    MODEL_ID,
    transformer=transformer,
    torch_dtype=torch.float16
)

# optional but recommended for T4
pipe.enable_model_cpu_offload()

# generate
image = pipe(
    "image of a dog sitting on a couch having a beer bottle ",
    num_inference_steps=28,
    guidance_scale=7.0,
    height=512,
    width=512
).images[0]

image

In [ ]:
# evaluation metrics

import os, gc, random, math
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm


try:
    import clip as openai_clip
except ImportError:
    %pip install -q git+https://github.com/openai/CLIP.git
    import clip as openai_clip
from diffusers import StableDiffusion3Pipeline
from torchvision import transforms
import torch.nn.functional as F

FINETUNED_DIR   = "/content/sd3-finetuned"          # your OUTPUT_DIR
BASE_MODEL_ID   = "stabilityai/stable-diffusion-3-medium-diffusers"
KAGGLE_DATASET  = "yashraj2311/chimera-dataset"      # for real reference images
OUTPUT_DIR      = "./eval_results"
DEVICE          = "cuda" if torch.cuda.is_available() else "cpu"
SEED            = 42
NUM_IMAGES_GEN  = 6   # images generated per prompt (keep low for VRAM)
INFERENCE_STEPS = 28
GUIDANCE_SCALE  = 7.0
RESOLUTION      = 512

# Prompts to test
TEST_PROMPTS = {
    "cat_simple":    "a photo of a cat",
    "cat_sitting":   "a cat sitting on a couch",
    "cat_outdoor":   "a cat outdoors in a garden",
    # control concepts — should be UNAFFECTED
    "dog_simple":    "a photo of a dog",
    "car_simple":    "a photo of a car",
    # what Nightshade tries to make "cat" look like
    # (common target is often 'dog' or something semantically different)
    "nightshade_target": "a photo of a dog",   # change if you know your target
}

os.makedirs(OUTPUT_DIR, exist_ok=True)

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def free(*objs):
    for o in objs: del o
    gc.collect()
    torch.cuda.empty_cache()

def load_pipeline(model_path, dtype=torch.float16):
    print(f"Loading pipeline from: {model_path}")
    pipe = StableDiffusion3Pipeline.from_pretrained(
        model_path, torch_dtype=dtype, safety_checker=None
    ).to(DEVICE)
    pipe.set_progress_bar_config(disable=True)
    return pipe

@torch.no_grad()
def generate_images(pipe, prompt, n=NUM_IMAGES_GEN, seed=SEED):
    images = []
    for i in range(n):
        g = torch.Generator(device=DEVICE).manual_seed(seed + i)
        img = pipe(
            prompt,
            num_inference_steps=INFERENCE_STEPS,
            guidance_scale=GUIDANCE_SCALE,
            generator=g,
            height=RESOLUTION,
            width=RESOLUTION,
        ).images[0]
        images.append(img)
    return images

_clip_model = None
_clip_preprocess = None

def get_clip():
    global _clip_model, _clip_preprocess
    if _clip_model is None:
        _clip_model, _clip_preprocess = openai_clip.load("ViT-B/32", device=DEVICE)
        _clip_model.eval()
    return _clip_model, _clip_preprocess

@torch.no_grad()
def clip_image_embed(images):
    """List[PIL.Image] → [N, 512] normalised float32"""
    model, preprocess = get_clip()
    tensors = torch.stack([preprocess(img) for img in images]).to(DEVICE)
    feats = model.encode_image(tensors).float()
    return F.normalize(feats, dim=-1)

@torch.no_grad()
def clip_text_embed(texts):
    """List[str] → [N, 512] normalised float32"""
    model, _ = get_clip()
    tokens = openai_clip.tokenize(texts, truncate=True).to(DEVICE)
    feats = model.encode_text(tokens).float()
    return F.normalize(feats, dim=-1)

def clip_score(images, prompt):
    """Mean cosine similarity between images and a text prompt (0→1)."""
    img_emb  = clip_image_embed(images)               # [N,512]
    txt_emb  = clip_text_embed([prompt])              # [1,512]
    sims     = (img_emb @ txt_emb.T).squeeze(-1)      # [N]
    return sims.mean().item(), sims.cpu().numpy()

def pairwise_sim(emb_a, emb_b):
    """Mean cosine sim between two sets of embeddings."""
    # emb_a [N,512], emb_b [M,512]
    sim_matrix = emb_a @ emb_b.T   # [N,M]
    return sim_matrix.mean().item()

def load_real_images_from_kaggle(n_per_class=30):
    """Pull real cat / dog / car images from the Kaggle dataset."""
    try:
        import kagglehub
        base = Path(kagglehub.dataset_download(KAGGLE_DATASET))
    except Exception as e:
        print(f"[WARN] Could not load Kaggle dataset: {e}")
        return {}

    to_pil = transforms.Compose([
        transforms.Resize(224),
        transforms.CenterCrop(224),
    ])

    results = {}
    for concept in ["cat", "dog", "car"]:
        imgs = sorted(base.rglob(f"{concept}*"))
        imgs = [p for p in imgs if p.suffix.lower() in {".jpg",".jpeg",".png"}]
        random.shuffle(imgs)
        pil_imgs = []
        for p in imgs[:n_per_class]:
            try:
                pil_imgs.append(to_pil(Image.open(p).convert("RGB")))
            except Exception:
                pass
        results[concept] = pil_imgs
        print(f"  Loaded {len(pil_imgs)} real {concept} images")
    return results

def run_evaluation():
    seed_everything(SEED)
    results = {}   # concept → dict of metrics

    # Real reference images
    print("\n[1/4] Loading real reference images …")
    real_images = load_real_images_from_kaggle(n_per_class=30)

    # Pre-compute real embeddings
    real_embeds = {}
    for concept, imgs in real_images.items():
        if imgs:
            real_embeds[concept] = clip_image_embed(imgs)

    # Generate from fine-tuned model
    print("\n[2/4] Generating images from fine-tuned model …")
    ft_images   = {}   # prompt_key → List[PIL]
    ft_embeds   = {}

    pipe = load_pipeline(FINETUNED_DIR)

    for key, prompt in TEST_PROMPTS.items():
        print(f"  Generating: '{prompt}'")
        imgs = generate_images(pipe, prompt)
        ft_images[key]  = imgs
        ft_embeds[key]  = clip_image_embed(imgs)

        # Save generated images
        out_dir = Path(OUTPUT_DIR) / key
        out_dir.mkdir(exist_ok=True)
        for i, img in enumerate(imgs):
            img.save(out_dir / f"gen_{i:02d}.png")

    free(pipe)

    # Generate from BASE model (baseline comparison)
    print("\n[3/4] Generating baseline images from base model …")
    base_images = {}
    base_embeds = {}

    base_pipe = load_pipeline(BASE_MODEL_ID)

    for key, prompt in TEST_PROMPTS.items():
        print(f"  Baseline: '{prompt}'")
        imgs = generate_images(base_pipe, prompt)
        base_images[key]  = imgs
        base_embeds[key]  = clip_image_embed(imgs)

    free(base_pipe)

    # Compute metrics
    print("\n[4/4] Computing metrics …")

    for key, prompt in TEST_PROMPTS.items():
        ft_score,   ft_sims   = clip_score(ft_images[key],   prompt)
        base_score, base_sims = clip_score(base_images[key], prompt)

        # Similarity to real cat images (concept drift)
        ft_vs_real_cat   = pairwise_sim(ft_embeds[key],   real_embeds.get("cat",   ft_embeds[key]))
        base_vs_real_cat = pairwise_sim(base_embeds[key], real_embeds.get("cat",   base_embeds[key]))

        # Similarity to real dog images (target confusion — did "cat" drift toward "dog"?)
        ft_vs_real_dog   = pairwise_sim(ft_embeds[key],   real_embeds.get("dog",   ft_embeds[key]))
        base_vs_real_dog = pairwise_sim(base_embeds[key], real_embeds.get("dog",   base_embeds[key]))

        # Intra-set diversity (lower = more mode collapse)
        ft_div   = 1 - pairwise_sim(ft_embeds[key],   ft_embeds[key])
        base_div = 1 - pairwise_sim(base_embeds[key], base_embeds[key])

        results[key] = {
            "prompt":           prompt,
            "ft_clip_score":    ft_score,
            "base_clip_score":  base_score,
            "clip_score_drop":  base_score - ft_score,
            "ft_vs_real_cat":   ft_vs_real_cat,
            "base_vs_real_cat": base_vs_real_cat,
            "ft_vs_real_dog":   ft_vs_real_dog,
            "base_vs_real_dog": base_vs_real_dog,
            "ft_diversity":     ft_div,
            "base_diversity":   base_div,
            "ft_sims":          ft_sims,
            "base_sims":        base_sims,
        }
        print(f"\n  [{key}]")
        print(f"    CLIP score  — fine-tuned: {ft_score:.4f}  |  base: {base_score:.4f}  |  drop: {base_score-ft_score:+.4f}")
        print(f"    vs real cat — fine-tuned: {ft_vs_real_cat:.4f}  |  base: {base_vs_real_cat:.4f}")
        print(f"    vs real dog — fine-tuned: {ft_vs_real_dog:.4f}  |  base: {base_vs_real_dog:.4f}")

    
    
    plot_results(results, ft_images, base_images, real_images)
    save_summary_csv(results)
    print(f"\n✅ Evaluation complete. Results saved to: {OUTPUT_DIR}")


def plot_results(results, ft_images, base_images, real_images):
    keys   = list(results.keys())
    labels = [k.replace("_", "\n") for k in keys]

    # Fig 1: CLIP Score comparison bar chart
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle("Nightshade Poison Evaluation — CLIP Metrics", fontsize=14, fontweight="bold")

    x = np.arange(len(keys))
    w = 0.35

    # Panel A: CLIP score
    ax = axes[0]
    ax.bar(x - w/2, [results[k]["base_clip_score"] for k in keys], w,
           label="Base model", color="#4C9BE8", alpha=0.85)
    ax.bar(x + w/2, [results[k]["ft_clip_score"]   for k in keys], w,
           label="Fine-tuned",  color="#E85C4C", alpha=0.85)
    ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=8)
    ax.set_ylabel("CLIP Score (↑ better)"); ax.set_title("A. Text-Image Alignment")
    ax.legend(); ax.set_ylim(0, 0.4)
    ax.axhline(0.2, color="gray", linestyle="--", linewidth=0.8, alpha=0.5)
    # highlight cat prompts
    for i, k in enumerate(keys):
        if "cat" in k:
            ax.get_xticklabels()[i].set_color("red")
            ax.get_xticklabels()[i].set_fontweight("bold")

    # Panel B: Similarity to real cats
    ax = axes[1]
    ax.bar(x - w/2, [results[k]["base_vs_real_cat"] for k in keys], w,
           label="Base model", color="#4C9BE8", alpha=0.85)
    ax.bar(x + w/2, [results[k]["ft_vs_real_cat"]   for k in keys], w,
           label="Fine-tuned",  color="#E85C4C", alpha=0.85)
    ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=8)
    ax.set_ylabel("Cosine Similarity (↑ = more cat-like)")
    ax.set_title("B. Visual Similarity to Real Cats")
    ax.legend()
    for i, k in enumerate(keys):
        if "cat" in k:
            ax.get_xticklabels()[i].set_color("red")
            ax.get_xticklabels()[i].set_fontweight("bold")

    # Panel C: Similarity to real dogs (target confusion)
    ax = axes[2]
    ax.bar(x - w/2, [results[k]["base_vs_real_dog"] for k in keys], w,
           label="Base model", color="#4C9BE8", alpha=0.85)
    ax.bar(x + w/2, [results[k]["ft_vs_real_dog"]   for k in keys], w,
           label="Fine-tuned",  color="#E85C4C", alpha=0.85)
    ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=8)
    ax.set_ylabel("Cosine Similarity (↑ = more dog-like)")
    ax.set_title("C. Concept Confusion: Cat → Dog?")
    ax.legend()
    for i, k in enumerate(keys):
        if "cat" in k:
            ax.get_xticklabels()[i].set_color("red")
            ax.get_xticklabels()[i].set_fontweight("bold")

    plt.tight_layout()
    fig.savefig(f"{OUTPUT_DIR}/fig1_clip_metrics.png", dpi=150)
    plt.show()
    print(f"  Saved fig1_clip_metrics.png")

    # Fig 2: CLIP Score drop (poison effect magnitude)
    fig, ax = plt.subplots(figsize=(10, 4))
    drops   = [results[k]["clip_score_drop"] for k in keys]
    colors  = ["#E85C4C" if "cat" in k else "#888888" for k in keys]
    bars    = ax.bar(labels, drops, color=colors, alpha=0.85, edgecolor="white")
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_ylabel("CLIP Score Drop  (base − fine-tuned)  ↑ = more poisoned")
    ax.set_title("Poison Effect Magnitude per Concept\n(red = cat concepts)")
    for bar, val in zip(bars, drops):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                f"{val:+.3f}", ha="center", va="bottom", fontsize=9)
    plt.tight_layout()
    fig.savefig(f"{OUTPUT_DIR}/fig2_poison_magnitude.png", dpi=150)
    plt.show()
    print(f"  Saved fig2_poison_magnitude.png")

    # Fig 3: Per-image CLIP score distribution (violin)
    cat_keys  = [k for k in keys if "cat" in k]
    ctrl_keys = [k for k in keys if "cat" not in k and k != "nightshade_target"]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle("Per-Image CLIP Score Distribution", fontsize=13, fontweight="bold")

    for ax, subset, title in zip(axes,
                                  [cat_keys, ctrl_keys],
                                  ["Cat Prompts (poisoned concept)",
                                   "Control Prompts (should be unaffected)"]):
        base_data = [results[k]["base_sims"].tolist() for k in subset]
        ft_data   = [results[k]["ft_sims"].tolist()   for k in subset]
        xlabels   = [k.replace("_", "\n") for k in subset]
        positions = np.arange(1, len(subset)+1)

        vp_base = ax.violinplot(base_data, positions=positions - 0.18,
                                widths=0.3, showmedians=True)
        vp_ft   = ax.violinplot(ft_data,   positions=positions + 0.18,
                                widths=0.3, showmedians=True)

        for vp, color, label in [(vp_base, "#4C9BE8", "Base"),
                                  (vp_ft,   "#E85C4C", "Fine-tuned")]:
            for body in vp["bodies"]:
                body.set_facecolor(color); body.set_alpha(0.6)
            vp["cmedians"].set_color(color)
            vp["cbars"].set_color(color)
            vp["cmaxes"].set_color(color)
            vp["cmins"].set_color(color)

        from matplotlib.patches import Patch
        ax.legend(handles=[Patch(facecolor="#4C9BE8", label="Base"),
                            Patch(facecolor="#E85C4C", label="Fine-tuned")])
        ax.set_xticks(positions); ax.set_xticklabels(xlabels, fontsize=8)
        ax.set_ylabel("CLIP Score"); ax.set_title(title)

    plt.tight_layout()
    fig.savefig(f"{OUTPUT_DIR}/fig3_clip_distribution.png", dpi=150)
    plt.show()
    print(f"  Saved fig3_clip_distribution.png")

    # Fig 4: Visual grid — base vs fine-tuned
    cat_prompt_keys = [k for k in keys if "cat" in k]
    n_cols = NUM_IMAGES_GEN
    n_rows = len(cat_prompt_keys) * 2   # base row + ft row per prompt

    fig = plt.figure(figsize=(n_cols * 2.2, n_rows * 2.4))
    fig.suptitle("Generated Images: Base (top) vs Fine-tuned (bottom)\nCat prompts only",
                 fontsize=12, fontweight="bold")

    gs = gridspec.GridSpec(n_rows, n_cols, figure=fig, hspace=0.05, wspace=0.05)
    row = 0
    for key in cat_prompt_keys:
        for col, img in enumerate(base_images[key][:n_cols]):
            ax = fig.add_subplot(gs[row, col])
            ax.imshow(img); ax.axis("off")
            if col == 0:
                ax.set_ylabel(f"BASE\n{key}", fontsize=7, color="#4C9BE8",
                              rotation=0, labelpad=50, va="center")
        row += 1
        for col, img in enumerate(ft_images[key][:n_cols]):
            ax = fig.add_subplot(gs[row, col])
            ax.imshow(img); ax.axis("off")
            if col == 0:
                ax.set_ylabel(f"FT\n{key}", fontsize=7, color="#E85C4C",
                              rotation=0, labelpad=50, va="center")
        row += 1

    plt.savefig(f"{OUTPUT_DIR}/fig4_visual_comparison.png", dpi=120, bbox_inches="tight")
    plt.show()
    print(f"  Saved fig4_visual_comparison.png")

    # Fig 5: Radar chart — overall poison profile
    plot_radar(results)


def plot_radar(results):
    """Radar chart summarising 4 key dimensions for cat vs control."""
    cat_keys  = [k for k in results if "cat" in k]
    ctrl_keys = [k for k in results if "cat" not in k and k != "nightshade_target"]

    def avg(keys, metric):
        return np.mean([results[k][metric] for k in keys])

    # Metrics (all normalised so higher = worse poison for cat)
    dims = ["CLIP score", "Cat similarity", "Dog confusion", "Diversity"]

    def profile(keys):
        return [
            avg(keys, "ft_clip_score")  / 0.35,        # normalise to ~1
            avg(keys, "ft_vs_real_cat") / 0.80,
            avg(keys, "ft_vs_real_dog") / 0.80,
            avg(keys, "ft_diversity")   / 0.10,
        ]

    cat_vals  = profile(cat_keys)
    ctrl_vals = profile(ctrl_keys)

    angles = np.linspace(0, 2*np.pi, len(dims), endpoint=False).tolist()
    angles += angles[:1]
    cat_vals  = cat_vals  + cat_vals[:1]
    ctrl_vals = ctrl_vals + ctrl_vals[:1]

    fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))
    ax.plot(angles, cat_vals,  "o-", color="#E85C4C", linewidth=2, label="Cat (poisoned)")
    ax.fill(angles, cat_vals,  color="#E85C4C", alpha=0.2)
    ax.plot(angles, ctrl_vals, "o-", color="#4C9BE8", linewidth=2, label="Controls")
    ax.fill(angles, ctrl_vals, color="#4C9BE8", alpha=0.2)
    ax.set_thetagrids(np.degrees(angles[:-1]), dims)
    ax.set_title("Poison Profile Radar\n(higher = more cat-like / better quality)",
                 fontsize=11, fontweight="bold", pad=20)
    ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1))
    plt.tight_layout()
    fig.savefig(f"{OUTPUT_DIR}/fig5_radar.png", dpi=150)
    plt.show()
    print(f"  Saved fig5_radar.png")


def save_summary_csv(results):
    import csv
    fieldnames = [
        "key", "prompt",
        "ft_clip_score", "base_clip_score", "clip_score_drop",
        "ft_vs_real_cat", "base_vs_real_cat",
        "ft_vs_real_dog", "base_vs_real_dog",
        "ft_diversity",  "base_diversity",
    ]
    path = f"{OUTPUT_DIR}/metrics_summary.csv"
    with open(path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for key, m in results.items():
            writer.writerow({fn: m.get(fn, key if fn=="key" else "") for fn in fieldnames})
    print(f"  Saved metrics_summary.csv")


if __name__ == "__main__":
    run_evaluation()